In [23]:
import pandas as pd

In [24]:
def clean_data_for_clustering(df):
    """
    Prepares dataset for Unsupervised Clustering.
    Extracts product identity by combining name and categories.
    """
    # Select relevant columns
    df_clean = df[["name", "categories"]].copy()

    # Drop entries without a product name 
    df_clean = df_clean.dropna(subset=["name"])
    
    # Fill missing categories with an empty string so the combination doesn't fail
    df_clean["categories"] = df_clean["categories"].fillna("")

    # Combine name and categories into a single string 
    df_clean["product_info"] = (
        df_clean["name"].astype(str) + " " + df_clean["categories"].astype(str)
    ).str.lower()

    # Remove duplicates
    df_unique = df_clean.drop_duplicates(subset=["name"]).reset_index(drop=True)

    return df_unique["product_info"]

In [25]:
# Load and clean the data 
data = pd.read_csv("../data/1429_1.csv")
product_list = clean_data_for_clustering(data).to_list()
data.head()

/var/folders/gn/wg9xjv455tj2c2jns2j4b8yw0000gn/T/ipykernel_99377/1602615083.py:2: DtypeWarning: Columns (0: name, 1: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("../data/1429_1.csv")


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [26]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Embed cleaned strings 
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(product_list)

# Perform K-Means Clustering
num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

# Store results in dataframe for inspection
results = pd.DataFrame({
    "product_info": product_list,
    "cluster": cluster_labels
})

# Check the clusters 
for i in range(num_clusters):
    print(f"\n--- Cluster {i} Sample Items ---")
    print(results[results["cluster"] == i]["product_info"].head(3).values)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8903.82it/s]



--- Cluster 0 Sample Items ---
<ArrowStringArray>
[                                                                                                                                                                                                                                                                                                                  'kindle oasis e-reader with leather charging cover - merlot, 6 high-resolution display (300 ppi), wi-fi - includes special offers,, ebook readers,kindle e-readers,computers & tablets,e-readers & accessories,e-readers',
                                                                                                                                                                                                                                                              'kindle oasis e-reader with leather charging cover - black, 6 high-resolution display (300 ppi), wi-fi - includes special offers,, fire tablets,tablets,computers & table

In [27]:
# Dictionary of unique products
name_to_cluster = dict(zip(results["product_info"], results["cluster"]))

# Recreate product info logic on entire dataset 
data["temp_id_key"] = (
    data["name"].astype(str) + " " + data["categories"].astype(str)
).str.lower()

# Map using the new key
data["cluster_id"] = data["temp_id_key"].map(name_to_cluster)

# Clean up: Drop the temporary key and rows without a cluster
data = data.drop(columns=["temp_id_key"])

print(data[["name", "cluster_id"]].head())

                                                name  cluster_id
0  All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...         4.0
1  All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...         4.0
2  All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...         4.0
3  All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...         4.0
4  All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...         4.0


In [28]:
import re

def distill_cluster_v2(results_df, cluster_id):
    unique_names = results_df[results_df["cluster"] == cluster_id]["product_info"].unique()
    
    clean_set = set() # Set to avoid identical 'first 10 words'
    for name in unique_names:
        clean_name = re.sub(r"[^a-zA-Z0-9\s]", " ", name)
        clean_name = " ".join(clean_name.split()[:10]).lower()
        clean_set.add(clean_name)
    
    # Return up to 10 unique variations
    return "\n- ".join(list(clean_set)[:10])

print(distill_cluster_v2(results, 3.0))

amazon kindle fire hd 3rd generation 8gb amazon kindle fire
- kindle keyboard kindle keyboard electronics ebook readers accessories covers kindle
- amazon kindle lighted leather cover kindle keyboard electronics ebook readers
- all new kindle e reader black 6 glare free touchscreen
- amazon kindle lighted leather cover amazon kindle lighted leather cover


In [29]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def get_professional_label(distilled_text):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini", 
            messages=[
                {"role": "system", "content": "You are a retail data expert. Your task is to provide a single, professional 2-3 word category name that summarizes a list of products. Avoid being too specific (e.g., use 'Tableware' instead of 'Plates'). Return ONLY the category name."},
                {"role": "user", "content": f"Product List:\n{distilled_text}"}
            ],
            temperature=0 
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return "Uncategorized"

print(get_professional_label(distill_cluster_v2(results, 1.0)))

Smart Devices


In [30]:
# Create mapping for clustering labels 
c_labels_map = {}
existing_labels = set()
unique_clusters = [c for c in data["cluster_id"].unique() if pd.notna(c)]

for cluster in unique_clusters:
    cluster_key = int(cluster)
    label = get_professional_label(distill_cluster_v2(results, cluster_key))
    if label in existing_labels:
        label = f"{label} (Group {cluster_key})"
    
    c_labels_map[cluster_key] = label
    existing_labels.add(label)

In [31]:
data["cluster"] = data["cluster_id"].map(c_labels_map).fillna("Miscellaneous")
data.head()

,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,cluster_id,cluster
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter,4.0,Tablets
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman,4.0,Tablets
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ,4.0,Tablets
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks,4.0,Tablets
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42,4.0,Tablets
